# Enterprise SIEM Lab: Security Log Monitoring and Alert Analysis
**IBM SkillsBuild Cyber Security 2026 — Student Project**

This beginner-friendly project demonstrates core SIEM ideas with a small, clearly labelled synthetic dataset. It parses security events, applies transparent detection rules, summarises alerts, and visualises findings.

> **Scope note:** This is a Python-based SIEM log-analysis prototype, not a deployed Wazuh cluster. The sample events are synthetic and must not be represented as real incidents.


## 1. Objectives
- Read structured security event logs.
- Identify repeated failed logins, successful logins after failures, and blocked connections.
- Prioritise alerts using simple rule-based severity.
- Summarise events and visualise alert counts.
- Explain limitations and possible next steps for a full SIEM such as Wazuh.


## 2. Setup
Install dependencies from `requirements.txt`, then run all notebook cells from top to bottom.

```bash
python -m pip install -r requirements.txt
jupyter notebook
```

The notebook uses synthetic data created below, so no external dataset download is required. This makes the project reproducible for a beginner. For a production deployment, connect a SIEM to authorised logs and validate detections before relying on them.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


## 3. Create a sample security-log dataset
The records below are fictional examples designed to demonstrate analysis logic.

In [ ]:
events = [
    {"timestamp": "2026-10-20 09:00:02", "source_ip": "192.0.2.10", "username": "alex", "event_type": "login_failed", "status": "failure", "host": "workstation-01"},
    {"timestamp": "2026-10-20 09:00:10", "source_ip": "192.0.2.10", "username": "alex", "event_type": "login_failed", "status": "failure", "host": "workstation-01"},
    {"timestamp": "2026-10-20 09:00:18", "source_ip": "192.0.2.10", "username": "alex", "event_type": "login_failed", "status": "failure", "host": "workstation-01"},
    {"timestamp": "2026-10-20 09:01:01", "source_ip": "192.0.2.10", "username": "alex", "event_type": "login_success", "status": "success", "host": "workstation-01"},
    {"timestamp": "2026-10-20 09:12:44", "source_ip": "198.51.100.23", "username": "unknown", "event_type": "connection_blocked", "status": "blocked", "host": "gateway-01"},
    {"timestamp": "2026-10-20 09:15:20", "source_ip": "192.0.2.25", "username": "sam", "event_type": "login_success", "status": "success", "host": "workstation-02"},
    {"timestamp": "2026-10-20 09:18:02", "source_ip": "198.51.100.23", "username": "unknown", "event_type": "connection_blocked", "status": "blocked", "host": "gateway-01"},
    {"timestamp": "2026-10-20 09:20:51", "source_ip": "192.0.2.30", "username": "jordan", "event_type": "login_failed", "status": "failure", "host": "workstation-03"},
]
df = pd.DataFrame(events)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df


## 4. Explore the data

In [ ]:
print(f"Total events: {len(df)}")
print("\nEvent types:")
print(df["event_type"].value_counts())
print("\nEvents by host:")
print(df["host"].value_counts())


## 5. Apply simple detection rules
Rules used in this demonstration:
1. **Repeated failed logins:** at least 3 failures from one source IP for one username.
2. **Successful login after repeated failures:** a success from the same source IP and username occurs after the repeated failures.
3. **Repeated blocked connections:** at least 2 blocked events from the same source IP.

These are educational example rules. Real detection needs context, tuning, time windows, and false-positive review.


In [ ]:
alerts = []

# Rule 1: repeated failed logins grouped by source IP and username
failed = df[df["event_type"] == "login_failed"]
for (ip, username), group in failed.groupby(["source_ip", "username"]):
    if len(group) >= 3:
        alerts.append({
            "timestamp": group["timestamp"].max(),
            "source_ip": ip,
            "username": username,
            "alert": "Repeated failed logins",
            "severity": "High",
            "evidence": f"{len(group)} failed login events"
        })

# Rule 2: successful login after repeated failures, for same IP/user
for (ip, username), group in failed.groupby(["source_ip", "username"]):
    if len(group) >= 3:
        later_success = df[
            (df["source_ip"] == ip) &
            (df["username"] == username) &
            (df["event_type"] == "login_success") &
            (df["timestamp"] > group["timestamp"].max())
        ]
        if not later_success.empty:
            alerts.append({
                "timestamp": later_success["timestamp"].min(),
                "source_ip": ip,
                "username": username,
                "alert": "Successful login after repeated failures",
                "severity": "Critical",
                "evidence": "Successful login followed at least 3 failed logins"
            })

# Rule 3: repeated blocked connections
blocked = df[df["event_type"] == "connection_blocked"]
for ip, group in blocked.groupby("source_ip"):
    if len(group) >= 2:
        alerts.append({
            "timestamp": group["timestamp"].max(),
            "source_ip": ip,
            "username": "-",
            "alert": "Repeated blocked connections",
            "severity": "Medium",
            "evidence": f"{len(group)} blocked connection events"
        })

alerts_df = pd.DataFrame(alerts)
if not alerts_df.empty:
    alerts_df = alerts_df.sort_values("timestamp").reset_index(drop=True)
alerts_df


## 6. Alert summary

In [ ]:
if alerts_df.empty:
    print("No alerts were generated by the demonstration rules.")
else:
    print("Alerts by severity:")
    print(alerts_df["severity"].value_counts())
    print("\nAlerts by rule:")
    print(alerts_df["alert"].value_counts())


## 7. Visualise results

In [ ]:
if not alerts_df.empty:
    severity_counts = alerts_df["severity"].value_counts()
    severity_counts.plot(kind="bar")
    plt.title("Alerts by Severity")
    plt.xlabel("Severity")
    plt.ylabel("Number of Alerts")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    rule_counts = alerts_df["alert"].value_counts()
    rule_counts.plot(kind="bar")
    plt.title("Alerts by Detection Rule")
    plt.xlabel("Detection Rule")
    plt.ylabel("Number of Alerts")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()


## 8. Export results
The notebook writes the sample events and generated alerts to CSV files in the current working directory.


In [ ]:
output_dir = Path("siem_output")
output_dir.mkdir(exist_ok=True)
df.to_csv(output_dir / "sample_security_events.csv", index=False)
alerts_df.to_csv(output_dir / "detected_alerts.csv", index=False)
print(f"Saved outputs to: {output_dir.resolve()}")


## 9. Interpretation
The rules identify patterns that may deserve investigation. Repeated failed logins can indicate password guessing, a success following repeated failures can warrant review, and repeated blocked connections can indicate repeated attempts to reach a protected service. These patterns alone do not prove malicious activity.

The IP ranges in this synthetic dataset are documentation-only example ranges, not indicators attributed to real attackers.


## 10. Limitations and future improvements
- The dataset is small, synthetic, and not representative of enterprise traffic.
- Rules are simple and do not include a rolling time window or user/asset context.
- The prototype does not collect logs from endpoints, a firewall, or a live SIEM.
- It does not block traffic or take automated response actions.
- Future work could ingest authorised Windows/Linux/firewall logs, add time-window rules, test on a public labelled dataset, measure precision/recall, and integrate with Wazuh in a controlled lab.


## 11. Conclusion
This project demonstrates the basic SIEM workflow: collect events, analyse patterns, generate alerts, summarise results, and support investigation. It is an educational prototype and should not be used as a production security monitoring system.
